In [ ]:
from prep_car import update_combined_sheet
from pathlib import Path

# 루트 디렉토리 경로를 설정합니다. (현재 파일의 부모 폴더('utils')의 부모 폴더)
ROOT_DIR = Path.cwd().parent

input_dir = ROOT_DIR / 'data' / 'car' # car폴더의 엑셀을 찾아서
output_dir = ROOT_DIR / 'data'        # data폴더에 통합 데이터 만들기
output_csv_path = output_dir / f"car_all_sheet.csv" # 전처리 완료 통합 데이터

update_combined_sheet(input_dir, output_csv_path)


📁 파일 처리 시작: c:\Users\user\Desktop\rag\data\car\의학계열 입결정리_2026학년도.xlsx
-> '입결(의대)' 시트 처리 완료.
-> '입결(치대)' 시트 처리 완료.
-> '입결(약대)' 시트 처리 완료.
-> '입결(수의예)' 시트 처리 완료.
-> '입결(한의예)' 시트 처리 완료.
-> '입결(문디컬)' 시트 처리 완료.
총 1320개의 데이터 행을 처리했습니다.
새로운 통합 파일 'c:\Users\user\Desktop\rag\data\car_all_sheet.csv'을 생성합니다.


In [ ]:
import pandas as pd
import re

df = pd.read_csv("의학계열 입결정리_2026학년도 - 입결(의대).csv", header=None)
df_column_merge = df.copy()
# 병합된 경우, 왼쪽열의 값을 채워넣기
df_column_merge.iloc[0:2, :] = df_column_merge.iloc[0:2, :].ffill(axis=1)

# null은 ''로 채우고, 구분자 '_'로 합치기
new_columns = df_column_merge.iloc[0:3, :].fillna('').apply(lambda x: '_'.join(x).strip())

# 해당 값을 칼럼 이름에 넣고, 기존 행 삭제
df_column_merge.columns = new_columns

# 집계열 제거
df_cleaned = df_column_merge.loc[:, ~df_column_merge.columns.str.contains('증감|평균|대비')]
df_cleaned = df_cleaned.drop(index=[0, 1, 2])

# 집계행 제거
df_cleaned = df_cleaned.dropna(subset=[df_cleaned.columns[1]])
df_cleaned = df_cleaned.reset_index(drop=True)

df_cleaned.columns = df_cleaned.columns.str.replace('\n학년도', '', regex=False)
df_cleaned.columns = df_cleaned.columns.str.replace('학년도', '', regex=False)
df_cleaned.columns = df_cleaned.columns.str.rstrip('_')

df_cleaned.columns = df_cleaned.columns.str.replace(r'실질경쟁률_(\d{4})_(.*)', r'\2_\1', regex=True) \
                                   .str.replace(r'입결\((\d{4}).*?\)_어디가_(.*)', r'입결\2_\1', regex=True)


##################################

# 1. 'melt'를 사용해 연도별로 흩어진 데이터를 긴 형태로 변환
id_vars = ['세부', '대학', '전형', '학과']
df_long = df_cleaned.melt(
    id_vars=id_vars,
    var_name='속성',
    value_name='값'
)

# 2. '속성' 열에서 '측정항목'과 '년도'를 분리
# 예: '모집인원_2024' -> '모집인원', '2024'
df_long[['측정항목', '년도']] = df_long['속성'].str.rsplit('_', n=1, expand=True)

# 3. 'pivot_table'을 사용해 측정항목들을 다시 열로 펼치기
df_tidy = df_long.pivot_table(
    index=id_vars + ['년도'],
    columns='측정항목',
    values='값',
    aggfunc='first'
).reset_index()
df_tidy.columns.name = None # 불필요한 인덱스 이름 제거

# --- 여기서부터 '전형방법' 열 분리 시작 ---

# 4. '전형방법' 열의 NaN 값을 빈 문자열로 바꿔 에러 방지
df_tidy['전형방법'] = df_tidy['전형방법'].fillna('')

# 5. 정규표현식으로 각 단계별 정보 추출
df_tidy['1단계'] = df_tidy['전형방법'].str.extract(r'\[1\](.*?)(?:\n|$)')
df_tidy['2단계'] = df_tidy['전형방법'].str.extract(r'\[2\](.*?)(?:\n|$)')
df_tidy['일괄전형'] = df_tidy['전형방법'].str.extract(r'\[일괄\](.*)')

# 6. 원본 '전형방법' 열 삭제 및 불필요한 공백 제거
df_final = df_tidy.drop(columns=['전형방법'])
df_final['1단계'] = df_final['1단계'].str.strip()
df_final['2단계'] = df_final['2단계'].str.strip()
df_final['일괄전형'] = df_final['일괄전형'].str.strip()

#############################


# --- 사전 준비 ---
# 1. '일괄전형' 내용을 '1단계' 열과 통합하여 '1단계_통합' 열 생성
#    '1단계'에 값이 있으면 그 값을, 없으면(NaN) '일괄전형' 값을 사용
df_final['1단계_통합'] = df_final['1단계'].fillna(df_final['일괄전형'])


# 2. 전형 단계의 세부 내용을 분해하는 함수 정의
def parse_stage_details(stage_string):
    # 입력값이 비어있으면 빈 Series 반환
    if pd.isna(stage_string):
        return pd.Series({'방법1': None, '비중1': None, '방법2': None, '비중2': None})

    # '+' 기준으로 1차, 2차 요소를 분리 (예: '교과70', '면접30')
    components = stage_string.split('+')
    
    parsed_data = {}
    for i, comp in enumerate(components, 1):
        if i > 2: continue # 최대 2개 요소까지만 처리
        
        # 정규표현식으로 '문자'(방법)와 '숫자'(비중)를 분리
        match = re.search(r'([^\d]+)(\d+)', comp)
        if match:
            parsed_data[f'방법{i}'] = match.group(1).strip()
            parsed_data[f'비중{i}'] = int(match.group(2).strip())
        else: # '서류100' 처럼 숫자만 있는 경우
             parsed_data[f'방법{i}'] = comp.strip()
             parsed_data[f'비중{i}'] = None

    return pd.Series(parsed_data)


# 3. 함수를 '1단계_통합' 열과 '2단계' 열에 각각 적용
stage1_details = df_final['1단계_통합'].apply(parse_stage_details)
stage2_details = df_final['2단계'].apply(parse_stage_details)

# 4. 생성된 열들의 이름을 최종적인 형태로 변경
stage1_details = stage1_details.rename(columns={
    '방법1': '1-1차', '비중1': '1-1비중',
    '방법2': '1-2차', '비중2': '1-2비중'
})
stage2_details = stage2_details.rename(columns={
    '방법1': '2-1차', '비중1': '2-1비중',
    '방법2': '2-2차', '비중2': '2-2비중'
})

# 5. 기존 데이터프레임에 새로 생성된 열들을 합치기
df_structured = pd.concat([df_final, stage1_details, stage2_details], axis=1)

# 6. 중간 과정에서 사용했던 불필요한 열들 삭제
df_structured = df_structured.drop(columns=['1단계', '2단계', '일괄전형', '1단계_통합'])

# 최종 결과 확인
df_structured[df_structured['대학']=='고신대'].iloc[:10,14:]
df_structured


,세부,대학,전형,학과,년도,50%-70% 컷차이,경쟁률,모집인원,수능최저,입결50%,...,충원+최저충족,충원율,1-1차,1-1비중,1-2차,1-2비중,2-1차,2-1비중,2-2차,2-2비중
0,일반교과,가천대,학생부우수자,의예과,2023,-,-,5,-,-,...,-,-,None,NaN,None,NaN,None,NaN,None,NaN
1,일반교과,가천대,학생부우수자,의예과,2024,0.00,25.20,15,국수(미/기)영탐(2) 3개영역1등급,1.00,...,4.73,60%,교과,100.0,NaN,NaN,None,NaN,None,NaN
2,일반교과,가천대,학생부우수자,의예과,2025,-0.03,13.40,15,국수(미/기)영탐(2) 3개영역1등급,1.05,...,2.51,60%,교과,100.0,NaN,NaN,None,NaN,None,NaN
3,일반교과,가천대,학생부우수자,의예과,2026,NaN,NaN,4,국수(미/기)영과(2) 3개영역 1등급,NaN,...,NaN,NaN,교과,100.0,NaN,NaN,None,NaN,None,NaN
4,일반교과,가톨릭관동대,일반,의예,2023,-0.03,16.00,9,"국수(미/기)영과(2) 3합 4, 소수점이하 버림",1.13,...,1.91,200%,None,NaN,None,NaN,None,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479,지역학종,충남대,종합II(지역인재),의예,2026,NaN,NaN,18,수(미/기) + 국영과(2) 상위2과목 3합5\n*수학필수,NaN,...,NaN,NaN,서류,100.0,NaN,NaN,서류,66.0,면접,33.0
480,지역학종,한림대,학종 (지역인재),의예,2023,-2.29,12.50,16,"국수(미/기)영과(2) 3합 4, 영어포함경우 영어는1",1.35,...,0.00,25%,None,NaN,None,NaN,None,NaN,None,NaN
481,지역학종,한림대,학종 (지역인재),의예,2024,-1.92,10.69,16,"국수(미/기)영과(2) 3합 4, 영어포함경우 영어는1",1.55,...,0.00,25%,서류,100.0,NaN,NaN,서류,70.0,면접,30.0
482,지역학종,한림대,학종 (지역인재),의예,2025,-0.34,12.68,19,X,1.28,...,11.03,15%,서류,100.0,NaN,NaN,서류,70.0,면접,30.0


In [26]:
import pandas as pd
import re
from pathlib import Path

# --- 1. 데이터 처리 함수 정의 (기존과 동일) ---

def prepare_header(df):
    """(엑셀 시트에서 읽어온) DataFrame의 여러 줄 헤더를 하나로 병합합니다."""
    # 헤더 정보가 있는 상위 2개 행에 대해 forward fill 적용
    df.iloc[0:2, :] = df.iloc[0:2, :].ffill(axis=1)
    
    # ✅ 수정됨: .astype(str)을 추가하여 모든 요소를 문자열로 변환
    new_columns = df.iloc[0:3, :].fillna('').apply(lambda x: '_'.join(x.astype(str)).strip().rstrip('_'))
    df.columns = new_columns
    
    # 헤더로 사용된 기존 0, 1, 2번 행 삭제
    df = df.drop(index=[0, 1, 2])
    return df

def clean_initial_data(df):
    """초기 데이터를 정리합니다 (불필요한 집계 행/열 제거)."""
    if '증감' in ''.join(df.columns):
        df = df.loc[:, ~df.columns.str.contains('증감|평균|대비')]
    
    df = df.dropna(subset=[df.columns[1]])
    df = df.reset_index(drop=True)
    return df

def transform_to_tidy(df):
    """Wide 형태의 데이터를 Tidy 형태로 변환합니다."""
    # ✅ '실질경쟁률_' 접두어 유지 + 연도는 뒤로 이동
    df.columns = (
        df.columns
          .str.replace(r'실질경쟁률_(\d{4})학년도_(.*)', r'\2_\1', regex=True)
          .str.replace(r'입결\((\d{4}).*?\)_어디가_(.*)', r'입결\2_\1', regex=True)
          .str.replace('학년도', '', regex=False)
          .str.replace('\n', '', regex=False)
    )

    id_vars = ['세부', '대학', '전형', '학과']
    df_long = df.melt(id_vars=id_vars, var_name='속성', value_name='값')

    # 마지막에 붙은 연도(_YYYY)를 '년도'로 분리. 연도 없는 항목(예: 3개년평균)은 NaN으로 둠
    extracted = df_long['속성'].str.extract(r'^(.*?)_?(\d{4})?$')
    df_long[['측정항목', '년도']] = extracted
    df_long['측정항목'] = df_long['측정항목'].fillna(df_long['속성'])

    # 피벗 (연도 없는 3개년평균도 그대로 남습니다)
    df_tidy = (
        df_long.pivot_table(
            index=id_vars + ['년도'],
            columns='측정항목',
            values='값',
            aggfunc='first'
        )
        .reset_index()
    )
    df_tidy.columns.name = None
    return df_tidy

def parse_stage_details(stage_string):
    """'전형방법' 문자열을 세부 요소(방법, 비중)로 분해하는 헬퍼 함수."""
    if pd.isna(stage_string):
        return pd.Series({'방법1': None, '비중1': None, '방법2': None, '비중2': None})
    
    components = stage_string.split('+')
    parsed_data = {}
    for i, comp in enumerate(components, 1):
        if i > 2: continue
        match = re.search(r'([^\d\s]+)\s*(\d+)', comp)
        if match:
            parsed_data[f'방법{i}'] = match.group(1).strip()
            parsed_data[f'비중{i}'] = int(match.group(2).strip())
        else:
            parsed_data[f'방법{i}'] = comp.strip()
            parsed_data[f'비중{i}'] = None
    return pd.Series(parsed_data)

def structure_admission_methods(df):
    """'전형방법' 열을 세부적인 구조화된 열로 변환합니다."""
    if '전형방법' not in df.columns:
        return df

    df['전형방법'] = df['전형방법'].fillna('')
    df['1단계'] = df['전형방법'].str.extract(r'\[1\](.*?)(?:\n|$)')[0].str.strip()
    df['2단계'] = df['전형방법'].str.extract(r'\[2\](.*?)(?:\n|$)')[0].str.strip()
    df['일괄전형'] = df['전형방법'].str.extract(r'\[일괄\](.*)')[0].str.strip()
    
    df['1단계_통합'] = df['1단계'].fillna(df['일괄전형'])
    
    stage1_details = df['1단계_통합'].apply(parse_stage_details).rename(columns={'방법1': '1-1차', '비중1': '1-1비중', '방법2': '1-2차', '비중2': '1-2비중'})
    stage2_details = df['2단계'].apply(parse_stage_details).rename(columns={'방법1': '2-1차', '비중1': '2-1비중', '방법2': '2-2차', '비중2': '2-2비중'})
    
    df_structured = pd.concat([df, stage1_details, stage2_details], axis=1)
    df_structured = df_structured.drop(columns=['전형방법', '1단계', '2단계', '일괄전형', '1단계_통합'])
    return df_structured

